# Ropedia Academy — B7 · 3D Gaussian Splatting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/B7.ipynb)

> **Renders by projecting (splatting) explicit 3D Gaussians and alpha-compositing them front-to-back — and attaches a per-Gaussian semantic label (the bridge to Track D).**
>
> 通过投影（泼溅）显式 3D 高斯并按前到后做 alpha 合成来渲染——并为每个高斯附加语义标签（通向 Track D 的桥梁）。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/B7

In [ ]:
import torch

# A scene = many explicit 3D GAUSSIANS (position, color, opacity, + a label).
N = 5
means   = torch.randn(N, 3) + torch.tensor([0, 0, 5.])
colors  = torch.rand(N, 3)
opacity = torch.rand(N)
labels  = torch.randint(0, 3, (N,))                  # a SEMANTIC tag per Gaussian (-> D4)
K = torch.tensor([[500,0,128.],[0,500,128.],[0,0,1.]])

splat = lambda m: ((K @ m)[:2] / (K @ m)[2], m[2])   # project a Gaussian center -> 2D, depth

# Render a pixel by FRONT-TO-BACK alpha blending (the differentiable rasterizer core).
pixel, T = torch.zeros(3), 1.0
for i in torch.argsort(means[:, 2]):                 # near -> far
    uv, z = splat(means[i]); a = opacity[i].item()
    pixel = pixel + T * a * colors[i]; T = T * (1 - a)
print("rasterized pixel:", pixel.round(decimals=3).tolist(), "| transmittance left:", round(T, 3))
print("each Gaussian also carries a label", labels.tolist(), "-> queryable semantic map")

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/B7
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks